In [1]:
!nvidia-smi

Sun May 10 12:56:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

**Problem 1**   
Counting DNA Nucleotides

In [6]:
%%writefile dna_count.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include <string.h>

__global__ void count_nucleotides(const char* d_str, int length, int* d_counts) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < length) {
        char base = d_str[idx];
        if (base == 'A') atomicAdd(&d_counts[0], 1);
        else if (base == 'C') atomicAdd(&d_counts[1], 1);
        else if (base == 'G') atomicAdd(&d_counts[2], 1);
        else if (base == 'T') atomicAdd(&d_counts[3], 1);
    }
}

int main() {
    const char* h_str = "AGCTTTTCATTCTGACTGCAACGGGCAATATGTCTCTGTGTGGATTAAAAAAAGAGTGTCTGATAGCAGC";
    int length = strlen(h_str);
    int h_counts[4] = {0, 0, 0, 0};

    char *d_str;
    int *d_counts;

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaMalloc((void**)&d_str, length * sizeof(char));
    cudaMalloc((void**)&d_counts, 4 * sizeof(int));

    cudaMemcpy(d_str, h_str, length * sizeof(char), cudaMemcpyHostToDevice);
    cudaMemcpy(d_counts, h_counts, 4 * sizeof(int), cudaMemcpyHostToDevice);

    int threadsPerBlock = 256;
    int blocksPerGrid = (length + threadsPerBlock - 1) / threadsPerBlock;

    cudaEventRecord(start);

    count_nucleotides<<<blocksPerGrid, threadsPerBlock>>>(d_str, length, d_counts);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop); // Wait for the GPU to finish

    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);

    cudaMemcpy(h_counts, d_counts, 4 * sizeof(int), cudaMemcpyDeviceToHost);

    printf("Results: %d %d %d %d\n", h_counts[0], h_counts[1], h_counts[2], h_counts[3]);
    printf("Time: %f ms\n", milliseconds);

    cudaFree(d_str);
    cudaFree(d_counts);
    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    return 0;
}

Overwriting dna_count.cu


In [7]:
!nvcc -o dna_count dna_count.cu
!./dna_count

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Results: 20 12 17 21
Time: 0.145856 ms


**Problem 2**                    
Transcribing DNA into RNA

In [8]:
%%writefile dna_count.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include <string.h>

__global__ void count_nucleotides(const char* d_str, int length, int* d_counts) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < length) {
        char base = d_str[idx];
        if (base == 'A') atomicAdd(&d_counts[0], 1);
        else if (base == 'C') atomicAdd(&d_counts[1], 1);
        else if (base == 'G') atomicAdd(&d_counts[2], 1);
        else if (base == 'T') atomicAdd(&d_counts[3], 1);
    }
}

int main() {
    const char* h_str = "AGCTTTTCATTCTGACTGCAACGGGCAATATGTCTCTGTGTGGATTAAAAAAAGAGTGTCTGATAGCAGC";
    int length = (int)strlen(h_str);
    int h_counts[4] = {0, 0, 0, 0};

    char *d_str;
    int *d_counts;

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaMalloc((void**)&d_str, length * sizeof(char));
    cudaMalloc((void**)&d_counts, 4 * sizeof(int));

    cudaMemcpy(d_str, h_str, length * sizeof(char), cudaMemcpyHostToDevice);
    cudaMemcpy(d_counts, h_counts, 4 * sizeof(int), cudaMemcpyHostToDevice);

    int threadsPerBlock = 256;
    int blocksPerGrid = (length + threadsPerBlock - 1) / threadsPerBlock;

    cudaEventRecord(start);
    count_nucleotides<<<blocksPerGrid, threadsPerBlock>>>(d_str, length, d_counts);
    cudaEventRecord(stop);

    cudaEventSynchronize(stop);

    float ms = 0;
    cudaEventElapsedTime(&ms, start, stop);

    cudaMemcpy(h_counts, d_counts, 4 * sizeof(int), cudaMemcpyDeviceToHost);

    printf("%d %d %d %d\n", h_counts[0], h_counts[1], h_counts[2], h_counts[3]);
    printf("Time: %f ms\n", ms);

    cudaFree(d_str);
    cudaFree(d_counts);
    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    return 0;
}

Overwriting dna_count.cu


In [9]:
!nvcc -o dna_count dna_count.cu
!./dna_count

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
20 12 17 21
Time: 0.210848 ms


**Problem 3**                                                                               
Calculating Protein Mass

In [1]:
%%writefile protein_weight.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include <string.h>

__device__ float get_weight(char aa) {
    switch (aa) {
        case 'A': return 71.03711f;  case 'C': return 103.00919f; case 'D': return 115.02694f;
        case 'E': return 129.04259f; case 'F': return 147.06841f; case 'G': return 57.02146f;
        case 'H': return 137.05891f; case 'I': return 113.08406f; case 'K': return 128.09496f;
        case 'L': return 113.08406f; case 'M': return 131.04049f; case 'N': return 114.04293f;
        case 'P': return 97.05276f;  case 'Q': return 128.05858f; case 'R': return 156.10111f;
        case 'S': return 87.03203f;  case 'T': return 101.04768f; case 'V': return 99.06841f;
        case 'W': return 186.07931f; case 'Y': return 163.06333f;
        default: return 0.0f;
    }
}

__global__ void calculate_weight(const char* d_str, int length, float* d_total) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < length) {
        atomicAdd(d_total, get_weight(d_str[idx]));
    }
}

int main() {
    const char* h_str = "SKADYEK";
    int length = (int)strlen(h_str);
    float h_total = 0.0f;

    char *d_str;
    float *d_total;

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaMalloc((void**)&d_str, length * sizeof(char));
    cudaMalloc((void**)&d_total, sizeof(float));

    cudaMemcpy(d_str, h_str, length * sizeof(char), cudaMemcpyHostToDevice);
    cudaMemcpy(d_total, &h_total, sizeof(float), cudaMemcpyHostToDevice);

    int threadsPerBlock = 256;
    int blocksPerGrid = (length + threadsPerBlock - 1) / threadsPerBlock;

    cudaEventRecord(start);
    calculate_weight<<<blocksPerGrid, threadsPerBlock>>>(d_str, length, d_total);
    cudaEventRecord(stop);

    cudaEventSynchronize(stop);

    float ms = 0;
    cudaEventElapsedTime(&ms, start, stop);

    cudaMemcpy(&h_total, d_total, sizeof(float), cudaMemcpyDeviceToHost);

    printf("%.3f\n", h_total);
    printf("Time: %f ms\n", ms);

    cudaFree(d_str);
    cudaFree(d_total);
    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    return 0;
}

Writing protein_weight.cu


In [2]:
!nvcc -o protein_weight protein_weight.cu
!./protein_weight

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
821.392
Time: 95.426270 ms


**Problem 4**                   
Translating RNA into Protein          

In [6]:
%%writefile rna_translation.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include <string.h>

__device__ char translate_codon(const char* s) {
    if (s[0] == 'U') {
        if (s[1] == 'U') return (s[2] == 'U' || s[2] == 'C') ? 'F' : 'L';
        if (s[1] == 'C') return 'S';
        if (s[1] == 'A') return (s[2] == 'U' || s[2] == 'C') ? 'Y' : 0;
        if (s[1] == 'G') return (s[2] == 'U' || s[2] == 'C') ? 'C' : (s[2] == 'A' ? 'W' : 0);
    } else if (s[0] == 'C') {
        if (s[1] == 'U') return 'L';
        if (s[1] == 'C') return 'P';
        if (s[1] == 'A') return (s[2] == 'U' || s[2] == 'C') ? 'H' : 'Q';
        if (s[1] == 'G') return 'R';
    } else if (s[0] == 'A') {
        if (s[1] == 'U') return (s[2] == 'G') ? 'M' : 'I';
        if (s[1] == 'C') return 'T';
        if (s[1] == 'A') return (s[2] == 'U' || s[2] == 'C') ? 'N' : 'K';
        if (s[1] == 'G') return (s[2] == 'U' || s[2] == 'C') ? 'S' : 'R';
    } else if (s[0] == 'G') {
        if (s[1] == 'U') return 'V';
        if (s[1] == 'C') return 'A';
        if (s[1] == 'A') return (s[2] == 'U' || s[2] == 'C') ? 'D' : 'E';
        if (s[1] == 'G') return 'G';
    }
    return 0;
}

__global__ void translate_kernel(const char* d_rna, char* d_prot, int prot_len) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < prot_len) {
        d_prot[i] = translate_codon(&d_rna[i * 3]);
    }
}

int main() {
    const char* h_rna = "AUGGCCAUGGCGCCCAGAACUGAGAUCAAUAGUACCCGUAUUAACGGGUGA";
    int rna_len = (int)strlen(h_rna);
    int prot_len = rna_len / 3;

    char *h_prot = (char*)malloc(prot_len + 1);
    char *d_rna, *d_prot;

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaMalloc((void**)&d_rna, rna_len * sizeof(char));
    cudaMalloc((void**)&d_prot, prot_len * sizeof(char));

    cudaMemcpy(d_rna, h_rna, rna_len * sizeof(char), cudaMemcpyHostToDevice);

    int threadsPerBlock = 256;
    int blocksPerGrid = (prot_len + threadsPerBlock - 1) / threadsPerBlock;

    cudaEventRecord(start);
    translate_kernel<<<blocksPerGrid, threadsPerBlock>>>(d_rna, d_prot, prot_len);
    cudaEventRecord(stop);

    cudaEventSynchronize(stop);

    float ms = 0;
    cudaEventElapsedTime(&ms, start, stop);

    cudaMemcpy(h_prot, d_prot, prot_len * sizeof(char), cudaMemcpyDeviceToHost);
    h_prot[prot_len] = '\0';

    for(int i = 0; i < prot_len; i++) {
        if(h_prot[i] == 0) break;
        printf("%c", h_prot[i]);
    }
    printf("\nTime: %f ms\n", ms);

    free(h_prot);
    cudaFree(d_rna);
    cudaFree(d_prot);
    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    return 0;
}

Overwriting rna_translation.cu


In [7]:
!nvcc -Wno-deprecated-gpu-targets -o rna_translation rna_translation.cu
!./rna_translation

MAMAPRTEINSTRINGW
Time: 17.600512 ms


**Problem 5 **                        
Counting Point Mutations

In [8]:
%%writefile hamming_distance.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include <string.h>

__global__ void compute_hamming(const char* s, const char* t, int length, int* distance) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < length) {
        if (s[idx] != t[idx]) {
            atomicAdd(distance, 1);
        }
    }
}

int main() {
    const char* h_s = "GAGCCTACTAACGGGAT";
    const char* h_t = "CATCGTAATGACGGCCT";

    int length = (int)strlen(h_s);
    int h_dist = 0;

    char *d_s, *d_t;
    int *d_dist;

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaMalloc((void**)&d_s, length * sizeof(char));
    cudaMalloc((void**)&d_t, length * sizeof(char));
    cudaMalloc((void**)&d_dist, sizeof(int));

    cudaMemcpy(d_s, h_s, length * sizeof(char), cudaMemcpyHostToDevice);
    cudaMemcpy(d_t, h_t, length * sizeof(char), cudaMemcpyHostToDevice);
    cudaMemcpy(d_dist, &h_dist, sizeof(int), cudaMemcpyHostToDevice);

    int threadsPerBlock = 256;
    int blocksPerGrid = (length + threadsPerBlock - 1) / threadsPerBlock;

    cudaEventRecord(start);
    compute_hamming<<<blocksPerGrid, threadsPerBlock>>>(d_s, d_t, length, d_dist);
    cudaEventRecord(stop);

    cudaEventSynchronize(stop);

    float ms = 0;
    cudaEventElapsedTime(&ms, start, stop);

    cudaMemcpy(&h_dist, d_dist, sizeof(int), cudaMemcpyDeviceToHost);

    printf("%d\n", h_dist);
    printf("Time: %f ms\n", ms);

    cudaFree(d_s);
    cudaFree(d_t);
    cudaFree(d_dist);
    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    return 0;
}

Writing hamming_distance.cu


In [9]:
!nvcc -Wno-deprecated-gpu-targets -o hamming_distance hamming_distance.cu
!./hamming_distance

7
Time: 13.001792 ms
